
# Physics-Informed PPO Ablation Notebook

This notebook is a continuation of the physics-informed PPO work. It is organized by **where the physics is injected**:

1. **State ablations**: add physics-derived state features such as tracking error, velocity error, acceleration error.
2. **Reward ablations**: keep the basic observation space fixed and change only physics-informed reward terms.
3. **Architecture ablations**: change temporal/architectural access to dynamics through observation stacking.
4. **Loss-function ablations**: keep deployment observations fixed and add training-only auxiliary dynamics losses.

Important status update: the poor-looking baseline was caused by training/protocol plumbing, not by PPO being intrinsically bad. The fair studies now use the continuous PPO baseline protocol: 500k requested PPO timesteps, 8 `subproc` environments, PPO `n_steps=256`, `batch_size=512`, `n_epochs=4`, and the same single train/eval signal JSONs.

Code fixes made for the rerun:

- Acceleration-state features `x_m_ddot` and `x_s_ddot` now use named top-level extractors instead of lambdas, which fixes the Windows `subproc` startup hang in F4/F5.
- TensorBoard logging is disabled by default unless `TELEOP_ENABLE_TENSORBOARD=1`, avoiding OneDrive/path writer failures during training.
- TensorBoard folders default to the run directory (`ppo/tb`) unless `TELEOP_TENSORBOARD_ROOT` is set.
- Focused-evaluation CSV and artifact checks use long-path-safe file access on Windows.
- Temporal-stack summaries now preserve the same protocol fields as the state/reward summaries, so fairness can be audited.

By default this notebook **does not run training**. It prints the exact commands so you can run the tests yourself, then rerun the analysis cells.


In [ ]:

from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "matlab_literal_env").exists():
    ROOT = Path(r"C:/Users/aha173/OneDrive - American University of Beirut/research/TeleopProject/TeleopDocs/TeleopWithRL")

RESULTS = ROOT / "matlab_literal_env" / "policy_gradient_experiments" / "results" / "dyn"
SPECS = ROOT / "matlab_literal_env" / "policy_gradient_experiments" / "results" / "specs"

# Safety flags. Leave NOTEBOOK_RUNS_STUDIES=False when you only want analysis.
NOTEBOOK_RUNS_STUDIES = False
SKIP_EXISTING = True
REFRESH_FOCUSED_EVAL = True
REQUIRE_COMPLETED_RESULTS = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
print(f"ROOT = {ROOT}")
print(f"RESULTS = {RESULTS}")
print(f"SPECS = {SPECS}")
print(f"NOTEBOOK_RUNS_STUDIES = {NOTEBOOK_RUNS_STUDIES}")



## Shared Gate Helpers

A result family is treated as usable only if the summary has every expected variant, each variant has the expected focused scenario count, and each focused folder contains per-scenario histories plus transparency-ratio plots. Low performance can still be a real result; missing rows, missing plots, non-finite metrics, or impossible budgets are treated as training/plumbing problems.

The helpers below are Windows long-path aware because several focused-evaluation scenario paths exceed the legacy 260-character limit.


In [ ]:

def _long_path(path: str | Path) -> str | Path:
    path = Path(path)
    if os.name != "nt":
        return path
    text = str(path.resolve())
    if text.startswith("\\\\?\\"):
        return text
    if text.startswith("\\\\"):
        return "\\\\?\\UNC\\" + text.lstrip("\\")
    return "\\\\?\\" + text


def file_exists(path: str | Path) -> bool:
    try:
        with open(_long_path(path), "rb"):
            return True
    except FileNotFoundError:
        return False


def dir_exists(path: str | Path) -> bool:
    try:
        with os.scandir(_long_path(path)):
            return True
    except (FileNotFoundError, NotADirectoryError):
        return False


def count_files(path: str | Path, suffix: str | None = None, recursive: bool = False) -> int:
    path = Path(path)
    if not dir_exists(path):
        return 0
    if recursive:
        total = 0
        for _, _, files in os.walk(_long_path(path)):
            total += sum(1 for name in files if suffix is None or name.endswith(suffix))
        return total
    with os.scandir(_long_path(path)) as entries:
        return sum(
            1
            for entry in entries
            if entry.is_file() and (suffix is None or entry.name.endswith(suffix))
        )


def load_json_file(path: str | Path) -> dict:
    with open(_long_path(path), "r", encoding="utf-8") as fh:
        return json.load(fh)


def load_summary(study_dir: Path) -> pd.DataFrame:
    csv_path = study_dir / "summary.csv"
    if not file_exists(csv_path):
        raise FileNotFoundError(csv_path)
    return pd.read_csv(_long_path(csv_path))


def load_summary_or_empty(study_dir: Path, expected_keys: list[str], label: str) -> pd.DataFrame:
    csv_path = study_dir / "summary.csv"
    if file_exists(csv_path):
        return load_summary(study_dir)
    display(Markdown(f"Missing `{csv_path}`. Run the `{label}` command in the setup cell, then rerun this notebook."))
    return pd.DataFrame({"key": expected_keys})


def artifact_audit(study_dir: Path, variant_dirs: list[str]) -> pd.DataFrame:
    rows = []
    for key in variant_dirs:
        focused = study_dir / key / "focused_eval"
        rows.append({
            "variant_dir": key,
            "metrics_csv": file_exists(focused / "focused_eval_metrics.csv"),
            "history_npz_count": count_files(focused / "histories", suffix=".npz"),
            "transparency_plot_count": count_files(focused, suffix="transparency_ratio.png", recursive=True),
            "scenario_png_count": count_files(focused / "plots" / "scenarios", suffix=".png"),
        })
    return pd.DataFrame(rows)


def summarize_gate(name: str, df: pd.DataFrame, expected_keys: list[str], audit: pd.DataFrame, expected_scenarios: int = 25) -> pd.DataFrame:
    present = set(df["key"].astype(str)) if "key" in df else set()
    rows = []
    rows.append({"gate": "expected_rows", "pass": len(df) == len(expected_keys), "detail": f"{len(df)} / {len(expected_keys)} rows"})
    rows.append({"gate": "expected_keys", "pass": set(expected_keys).issubset(present), "detail": ", ".join(k for k in expected_keys if k not in present) or "all present"})
    if "focused_scenario_count" in df:
        scenario_ok = pd.to_numeric(df["focused_scenario_count"], errors="coerce").fillna(-1).eq(expected_scenarios).all()
        rows.append({"gate": "focused_scenarios", "pass": bool(scenario_ok), "detail": f"expected {expected_scenarios} per row"})
    else:
        rows.append({"gate": "focused_scenarios", "pass": False, "detail": "summary missing focused_scenario_count"})
    metric_cols = [c for c in ["focused_tracking_rmse_mm", "focused_transparency_ratio_error_rmse", "focused_failure_rate"] if c in df.columns]
    if metric_cols:
        finite = np.isfinite(df[metric_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)).all()
        rows.append({"gate": "finite_metrics", "pass": bool(finite), "detail": ", ".join(metric_cols)})
    else:
        rows.append({"gate": "finite_metrics", "pass": False, "detail": "focused metrics missing"})
    if "focused_failure_rate" in df:
        failure = pd.to_numeric(df["focused_failure_rate"], errors="coerce")
        in_range = ((failure >= 0.0) & (failure <= 1.0)).all()
        rows.append({"gate": "failure_rate_range", "pass": bool(in_range), "detail": f"min={failure.min():.3g}, max={failure.max():.3g}"})
    rows.append({"gate": "history_artifacts", "pass": bool(audit["history_npz_count"].eq(expected_scenarios).all()), "detail": f"min={audit['history_npz_count'].min()}, max={audit['history_npz_count'].max()}"})
    rows.append({"gate": "transparency_plots", "pass": bool(audit["transparency_plot_count"].eq(expected_scenarios).all()), "detail": f"min={audit['transparency_plot_count'].min()}, max={audit['transparency_plot_count'].max()}"})
    rows.append({"gate": "scenario_pngs", "pass": bool(audit["scenario_png_count"].eq(expected_scenarios * 5).all()), "detail": f"min={audit['scenario_png_count'].min()}, max={audit['scenario_png_count'].max()}"})
    gate = pd.DataFrame(rows)
    display(Markdown(f"**{name} gate:** {'PASS' if gate['pass'].all() else 'CHECK'}"))
    display(gate)
    return gate


def display_sorted(df: pd.DataFrame, cols: list[str], sort_col: str) -> None:
    available = [c for c in cols if c in df.columns]
    if not available:
        display(df)
        return
    shown = df[available]
    if sort_col in shown.columns:
        shown = shown.sort_values(sort_col)
    display(shown)


def show_image(path: Path, title: str | None = None):
    if file_exists(path):
        if title:
            display(Markdown(f"**{title}**"))
        with open(_long_path(path), "rb") as fh:
            display(Image(data=fh.read()))
    else:
        display(Markdown(f"Missing image: `{path}`"))


In [ ]:

BASELINE_SUMMARY = RESULTS / "ppoSingleSignal" / "00b" / "ppo" / "l" / "summary.json"
TRAIN_SIGNAL_JSON = SPECS / "ppoSingleSignal_f0e321b1_train_signals.json"
EVAL_SIGNAL_JSON = SPECS / "ppoSingleSignal_f0e321b1_eval_signals.json"

# Pilot folders are kept only for historical reference; analysis below defaults to fair folders.
STATE_PILOT_DIR = RESULTS / "physics_informed_formulations_01"
REWARD_PILOT_DIR = RESULTS / "physics_reward_ablation_basic_obs_02"
ARCH_PILOT_DIR = RESULTS / "temporal_observation_stack_01"
LOSS_PILOT_DIR = RESULTS / "physics_auxiliary_gru_ppo_01"

STATE_DIR = RESULTS / "physics_informed_formulations_02_fair_500k"
REWARD_DIR = RESULTS / "physics_reward_ablation_basic_obs_03_fair_500k"
ARCH_DIR = RESULTS / "temporal_observation_stack_02_fair_500k"
LOSS_DIR = RESULTS / "physics_auxiliary_gru_ppo_02_fair_500k"

STATE_KEYS = ["F0_baseline", "F1_error_state_reward", "F2_error_dot_state_reward", "F3_error_ddot_state_reward", "F4_accel_state", "F5_accel_state_reward", "F6_effort_plus_delta_u"]
REWARD_KEYS = ["R0_e_only", "R1_e_edot", "R2_sliding", "R3_sliding_du", "R4_sliding_du_ddu", "R5_second_order", "R6_lyapunov", "R7_phase_direction", "R8_hf_deadzone"]
ARCH_KEYS = ["T0_pos_current", "T1_pos_stack3", "T2_pos_stack5", "T3_posvel_current", "T4_posvel_stack3"]
LOSS_KEYS = ["G0_gru_ppo", "G1_gru_prediction", "G2_gru_hidden_state", "G3_gru_prediction_hidden"]
LOSS_DIR_KEYS = ["G0", "G1p", "G2h", "G3ph"]

COMMON_SIGNAL_ARGS = [
    "--train-reset-options-json", str(TRAIN_SIGNAL_JSON),
    "--eval-reset-options-json", str(EVAL_SIGNAL_JSON),
]
SKIP_ARGS = ["--skip-existing"] if SKIP_EXISTING else []
REFRESH_ARGS = ["--refresh-focused-eval"] if REFRESH_FOCUSED_EVAL else []

STATE_CMD = ["python", "-B", "matlab_literal_env/policy_gradient_experiments/run_physics_informed_formulations.py", "--study-name", STATE_DIR.name, *SKIP_ARGS, *REFRESH_ARGS, "--ppo-device", "cpu", *COMMON_SIGNAL_ARGS]
REWARD_CMD = ["python", "-B", "matlab_literal_env/policy_gradient_experiments/run_physics_reward_ablation_basic_obs.py", "--study-name", REWARD_DIR.name, *SKIP_ARGS, *REFRESH_ARGS, "--ppo-device", "cpu", *COMMON_SIGNAL_ARGS]
ARCH_CMD = ["python", "-B", "matlab_literal_env/policy_gradient_experiments/run_temporal_observation_stack.py", "--study-name", ARCH_DIR.name, *SKIP_ARGS, *REFRESH_ARGS, "--ppo-device", "cpu", *COMMON_SIGNAL_ARGS]
LOSS_CMD = ["python", "-B", "matlab_literal_env/policy_gradient_experiments/run_auxiliary_gru_ppo.py", "--study-name", LOSS_DIR.name, "--reward-key", "R5_second_order", *SKIP_ARGS, "--device", "cpu", *COMMON_SIGNAL_ARGS]

COMMANDS = {
    "state fair 500k": STATE_CMD,
    "reward fair 500k": REWARD_CMD,
    "architecture fair 500k": ARCH_CMD,
    "loss fair 500k": LOSS_CMD,
}

for label, cmd in COMMANDS.items():
    print(label + ":", " ".join(cmd))

if NOTEBOOK_RUNS_STUDIES:
    for label, cmd in COMMANDS.items():
        print(f"\nRunning {label}...")
        subprocess.run(cmd, cwd=ROOT, check=True)
else:
    display(Markdown("Training is disabled in this notebook. To run the studies yourself, execute the printed commands in a terminal, or set `NOTEBOOK_RUNS_STUDIES = True` and rerun this cell."))



## 0. Training Protocol Audit

The main mismatch was not a physics term. Earlier plots mixed short pilot runs with different effective training settings, while the continuous PPO baseline notebook used a 500k-step, 8-environment PPO protocol on a fixed single train/eval signal pair.

This audit checks the fair state, reward, and temporal-architecture folders against the continuous PPO baseline protocol before interpreting ablations. The auxiliary GRU trainer is custom, so it is checked separately as a budget/artifact diagnostic rather than as an SB3 PPO protocol match.


In [ ]:

baseline_summary = load_json_file(BASELINE_SUMMARY) if file_exists(BASELINE_SUMMARY) else {}
baseline_hparams = baseline_summary.get("model_hyperparameters", {})

fair_protocol_dirs = {
    "state": (STATE_DIR, STATE_KEYS),
    "reward": (REWARD_DIR, REWARD_KEYS),
    "architecture": (ARCH_DIR, ARCH_KEYS),
}
fair_protocol_summaries = {
    family: load_summary(study_dir) if file_exists(study_dir / "summary.csv") else pd.DataFrame()
    for family, (study_dir, _) in fair_protocol_dirs.items()
}
reward_protocol_summary = fair_protocol_summaries["reward"]


def unique_or_list(series: pd.Series):
    values = series.dropna().unique().tolist()
    if len(values) == 1:
        return values[0]
    return values


baseline_protocol = {
    "total_timesteps": baseline_summary.get("total_timesteps"),
    "parallel_envs": baseline_summary.get("parallel_envs"),
    "resolved_vec_env_type": baseline_summary.get("resolved_vec_env_type", baseline_summary.get("vec_env_type")),
    "ppo_n_steps": baseline_summary.get("ppo_n_steps", baseline_hparams.get("n_steps")),
    "ppo_batch_size": baseline_summary.get("ppo_batch_size", baseline_hparams.get("batch_size")),
    "ppo_n_epochs": baseline_summary.get("ppo_n_epochs", baseline_hparams.get("n_epochs")),
    "train_signal_count": baseline_summary.get("train_signal_count"),
    "eval_signal_count": baseline_summary.get("eval_signal_count"),
}

protocol_rows = []
for family, df in fair_protocol_summaries.items():
    for field, baseline_value in baseline_protocol.items():
        observed = unique_or_list(df[field]) if field in df.columns and not df.empty else None
        protocol_rows.append({
            "family": family,
            "field": field,
            "continuous_ppo_baseline": baseline_value,
            "fair_study": observed,
            "match": baseline_value == observed,
        })
protocol_table = pd.DataFrame(protocol_rows)

protocol_gate_rows = [
    {"gate": "baseline_summary_found", "pass": file_exists(BASELINE_SUMMARY), "detail": str(BASELINE_SUMMARY)},
    {"gate": "train_signal_json_found", "pass": file_exists(TRAIN_SIGNAL_JSON), "detail": str(TRAIN_SIGNAL_JSON)},
    {"gate": "eval_signal_json_found", "pass": file_exists(EVAL_SIGNAL_JSON), "detail": str(EVAL_SIGNAL_JSON)},
]
for family, df in fair_protocol_summaries.items():
    family_table = protocol_table[protocol_table["family"] == family]
    actual_steps = unique_or_list(df["actual_train_timesteps"]) if "actual_train_timesteps" in df.columns and not df.empty else None
    protocol_gate_rows.extend([
        {"gate": f"{family}_summary_found", "pass": not df.empty, "detail": str(fair_protocol_dirs[family][0] / "summary.csv")},
        {"gate": f"{family}_protocol_matches_baseline", "pass": bool(not family_table.empty and family_table["match"].all()), "detail": "500k requested, 8 subproc envs, same PPO rollout/batch/epoch settings"},
        {"gate": f"{family}_actual_steps_recorded", "pass": actual_steps == 501760, "detail": f"actual_train_timesteps={actual_steps}"},
    ])
protocol_gate = pd.DataFrame(protocol_gate_rows)

performance_rows = []
if baseline_summary:
    performance_rows.append({
        "run": "continuous PPO baseline",
        "eval_tracking_rmse_mm": 1000.0 * baseline_summary.get("tracking_rmse_m", np.nan),
        "focused_tracking_rmse_mm": np.nan,
        "focused_transparency_ratio_error_rmse": np.nan,
    })
if not reward_protocol_summary.empty and "key" in reward_protocol_summary:
    r0_rows = reward_protocol_summary.loc[reward_protocol_summary["key"] == "R0_e_only"]
    if not r0_rows.empty:
        r0 = r0_rows.iloc[0]
        performance_rows.append({
            "run": "fair reward R0_e_only",
            "eval_tracking_rmse_mm": 1000.0 * r0.get("tracking_rmse_m", np.nan),
            "focused_tracking_rmse_mm": r0.get("focused_tracking_rmse_mm", np.nan),
            "focused_transparency_ratio_error_rmse": r0.get("focused_transparency_ratio_error_rmse", np.nan),
        })
performance_anchor = pd.DataFrame(performance_rows)

display(Markdown("**Protocol comparison against `51_ppo_continuous_baseline.ipynb`:**"))
display(protocol_table)
display(Markdown(f"**Protocol gate:** {'PASS' if protocol_gate['pass'].all() else 'CHECK'}"))
display(protocol_gate)
display(Markdown("**Baseline sanity anchor:** the old poor baseline was a protocol mismatch. The fair `R0_e_only` result should be close to the continuous PPO baseline, not a collapsed controller."))
display(performance_anchor)



## 1. State Ablations: Inject Physics Into the Observation

This family changes the policy input state and, in the original formulations, sometimes couples the state feature with a matching reward term. It answers: *does giving the policy physics-derived error coordinates make the controller easier to learn?*

Current fair folder: `physics_informed_formulations_02_fair_500k`.

Important fix: the F4/F5 acceleration-state variants previously hung under Windows `subproc` because `x_m_ddot` and `x_s_ddot` were lambda-backed feature extractors. They now use named top-level functions, so the fair state study can be rerun under the same 8-env protocol as the continuous PPO baseline.


In [ ]:

state_summary = load_summary_or_empty(STATE_DIR, STATE_KEYS, "state fair 500k")
state_audit = artifact_audit(STATE_DIR, STATE_KEYS)
state_gate = summarize_gate("State ablations", state_summary, STATE_KEYS, state_audit)

state_cols = ["key", "label", "obs_dim", "total_timesteps", "actual_train_timesteps", "parallel_envs", "resolved_vec_env_type", "state_features", "reward_terms", "focused_tracking_rmse_mm", "focused_failure_rate", "focused_transparency_ratio_error_rmse", "focused_transparency_ratio_within_20pct"]
display_sorted(state_summary, state_cols, "focused_tracking_rmse_mm")
display(state_audit)


In [ ]:
show_image(STATE_DIR / "summary_bars.png", "State ablation summary")
show_image(STATE_DIR / "learning_curves.png", "State ablation learning curves")
show_image(STATE_DIR / "F2_error_dot_state_reward" / "focused_eval" / "plots" / "scenarios" / "nominal_transparency_ratio.png", "State sample transparency ratio: F2 nominal")



## 2. Reward Ablations: Inject Physics Into the Reward

This family fixes the deployed observation to the basic sensor set:

`x_m, x_s, v_m, v_s, u_v`

Only the reward terms change. This is the cleanest test of whether physics-informed closed-loop error dynamics help beyond plain tracking error.

Current fair folder: `physics_reward_ablation_basic_obs_03_fair_500k`.

Every R0-R8 variant should be trained with the same 500k-step, 8-subproc-env, single-signal protocol as the continuous PPO baseline. The focused evaluation includes transparency-ratio metrics and per-scenario transparency plots for each variant.


In [ ]:

reward_summary = load_summary_or_empty(REWARD_DIR, REWARD_KEYS, "reward fair 500k")
reward_audit = artifact_audit(REWARD_DIR, REWARD_KEYS)
reward_gate = summarize_gate("Reward ablations", reward_summary, REWARD_KEYS, reward_audit)

reward_cols = ["key", "label", "total_timesteps", "actual_train_timesteps", "parallel_envs", "resolved_vec_env_type", "reward_terms", "focused_tracking_rmse_mm", "focused_failure_rate", "focused_transparency_ratio_error_rmse", "focused_transparency_ratio_within_20pct", "focused_mean_abs_delta2_u_v"]
display_sorted(reward_summary, reward_cols, "focused_tracking_rmse_mm")
display(reward_audit)


In [ ]:
show_image(REWARD_DIR / "reward_ablation_summary.png", "Reward ablation summary")
show_image(REWARD_DIR / "reward_ablation_training_curves.png", "Reward ablation training curves")
show_image(REWARD_DIR / "reward_ablation_group_heatmap.png", "Reward ablation group heatmap")
show_image(REWARD_DIR / "R5_second_order" / "focused_eval" / "plots" / "scenarios" / "nominal_transparency_ratio.png", "Reward sample transparency ratio: R5 nominal")



## 3. Architecture Ablations: Inject Physics Through Temporal Access

This family changes what the policy architecture can infer from short temporal windows. It does not add privileged physics variables; instead, stacking lets the policy infer local dynamics such as finite differences.

Current fair folder: `temporal_observation_stack_02_fair_500k`.

This runner now records the same protocol fields as the state/reward studies, so the notebook can check whether the temporal-stack comparison is fair before interpreting it. If this folder is missing, run the `architecture fair 500k` command from the setup cell.


In [ ]:

arch_summary = load_summary_or_empty(ARCH_DIR, ARCH_KEYS, "architecture fair 500k")
arch_audit = artifact_audit(ARCH_DIR, ARCH_KEYS)
arch_gate = summarize_gate("Architecture ablations", arch_summary, ARCH_KEYS, arch_audit)

arch_cols = ["key", "label", "obs_dim", "base_features", "lags", "total_timesteps", "actual_train_timesteps", "parallel_envs", "resolved_vec_env_type", "focused_tracking_rmse_mm", "focused_failure_rate", "focused_transparency_ratio_error_rmse", "focused_transparency_ratio_within_20pct"]
display_sorted(arch_summary, arch_cols, "focused_tracking_rmse_mm")
display(arch_audit)


In [ ]:
show_image(ARCH_DIR / "summary_bars.png", "Architecture ablation summary")
show_image(ARCH_DIR / "learning_curves.png", "Architecture learning curves")
show_image(ARCH_DIR / "T1_pos_stack3" / "focused_eval" / "plots" / "scenarios" / "nominal_transparency_ratio.png", "Architecture sample transparency ratio: T1 nominal")



## 4. Loss-Function Ablations: Inject Physics Through Training-Only Auxiliary Losses

This family keeps deployment observations basic and holds the reward fixed to `R5_second_order`. The ablation changes auxiliary losses in a GRU-PPO trainer:

- `G0`: GRU-PPO baseline.
- `G1`: prediction loss for next normalized position deltas.
- `G2`: hidden pneumatic state reconstruction loss.
- `G3`: both auxiliary losses.

Current fair folder: `physics_auxiliary_gru_ppo_02_fair_500k`.

This is a custom trainer rather than the SB3 PPO trainer. Treat it as a training-dynamics diagnostic until its budget coverage, focused failures, and artifacts pass the gates. If this folder is missing, run the `loss fair 500k` command from the setup cell.


In [ ]:

loss_summary = load_summary_or_empty(LOSS_DIR, LOSS_KEYS, "loss fair 500k")
loss_audit = artifact_audit(LOSS_DIR, LOSS_DIR_KEYS)
loss_gate = summarize_gate("Loss-function ablations", loss_summary, LOSS_KEYS, loss_audit)

loss_cols = ["key", "total_timesteps", "train_requested_timesteps", "train_budget_coverage", "train_finished_episodes", "train_completed_episodes", "prediction_loss_final", "hidden_state_loss_final", "focused_tracking_rmse_mm", "focused_failure_rate", "focused_transparency_ratio_error_rmse", "focused_transparency_ratio_within_20pct"]
display_sorted(loss_summary, loss_cols, "focused_tracking_rmse_mm")
display(loss_audit)


In [ ]:
show_image(LOSS_DIR / "auxiliary_gru_ppo_summary.png", "Loss-function ablation summary")
show_image(LOSS_DIR / "G2h" / "focused_eval" / "plots" / "scenarios" / "nominal_transparency_ratio.png", "Loss sample transparency ratio: G2 nominal")


## Cross-Family Interpretation

The tables below intentionally compare both performance and training health. A formulation with poor tracking but clean training/focused artifacts is a candidate negative result. A family with failed training gates, missing artifacts, or universal focused failures is treated as a training/plumbing issue before it is treated as a physics-formulation result.


In [ ]:

def _best_row(df: pd.DataFrame, metric: str) -> pd.Series | None:
    if df.empty or metric not in df.columns:
        return None
    numeric = pd.to_numeric(df[metric], errors="coerce")
    if numeric.notna().sum() == 0:
        return None
    return df.loc[numeric.idxmin()]


def family_best_rows() -> pd.DataFrame:
    rows = []
    for family, df, note in [
        ("state", state_summary, "fair 500k state-feature ablation; F4/F5 acceleration extractors fixed for Windows subproc"),
        ("reward", reward_summary, "fair 500k basic-observation reward ablation; matched to continuous PPO baseline protocol"),
        ("architecture", arch_summary, "fair 500k temporal-stack ablation when generated; otherwise pending"),
        ("loss", loss_summary, "fair-budget GRU-PPO auxiliary-loss diagnostic; custom trainer, compare cautiously"),
    ]:
        best_track = _best_row(df, "focused_tracking_rmse_mm")
        best_ratio = _best_row(df, "focused_transparency_ratio_error_rmse")
        failure = pd.to_numeric(df.get("focused_failure_rate", pd.Series(dtype=float)), errors="coerce")
        rows.append({
            "family": family,
            "best_tracking_key": None if best_track is None else best_track.get("key"),
            "best_tracking_rmse_mm": np.nan if best_track is None else best_track.get("focused_tracking_rmse_mm"),
            "best_ratio_key": None if best_ratio is None else best_ratio.get("key"),
            "best_ratio_error_rmse": np.nan if best_ratio is None else best_ratio.get("focused_transparency_ratio_error_rmse"),
            "max_failure_rate": failure.max() if not failure.empty else np.nan,
            "note": note,
        })
    return pd.DataFrame(rows)

family_comparison = family_best_rows()
display(family_comparison)

all_gates = pd.concat({
    "protocol": protocol_gate.set_index("gate"),
    "state": state_gate.set_index("gate"),
    "reward": reward_gate.set_index("gate"),
    "architecture": arch_gate.set_index("gate"),
    "loss": loss_gate.set_index("gate"),
}, names=["family", "gate"]).reset_index()
display(all_gates)



## Current Conclusions

- **The poor-results mismatch is addressed in code and protocol**: use the fair folders and commands above, not the old pilot folders, for final comparisons.
- **State training bug fixed**: F4/F5 acceleration-state variants now use named extractors for `x_m_ddot` and `x_s_ddot`, avoiding the Windows `subproc` startup hang.
- **Reward injection, basic observation**: this remains the cleanest physics-informed reward study because the observation is held fixed while reward terms change.
- **Temporal architecture**: the runner is ready for a fair 500k rerun and now records protocol metadata in `summary.csv` for audit.
- **Auxiliary GRU loss study**: run it as a custom-trainer diagnostic. Do not interpret poor auxiliary-GRU results as a final physics result unless its budget, focused failure rate, and artifact gates pass.
- **Transparency monitoring**: every fair focused evaluation is expected to produce 25 histories, 25 `*_transparency_ratio.png` plots, and 125 scenario PNGs per variant.


In [ ]:

artifact_rollup = pd.concat([
    artifact_audit(STATE_DIR, STATE_KEYS).assign(family="state fair 500k"),
    artifact_audit(REWARD_DIR, REWARD_KEYS).assign(family="reward fair 500k"),
    artifact_audit(ARCH_DIR, ARCH_KEYS).assign(family="architecture fair 500k"),
    artifact_audit(LOSS_DIR, LOSS_DIR_KEYS).assign(family="loss fair 500k"),
], ignore_index=True)
display(artifact_rollup[["family", "variant_dir", "metrics_csv", "history_npz_count", "transparency_plot_count", "scenario_png_count"]])

gate_status = pd.DataFrame([
    {"family": "protocol", "all_gates_pass": bool(protocol_gate["pass"].all())},
    {"family": "state", "all_gates_pass": bool(state_gate["pass"].all())},
    {"family": "reward", "all_gates_pass": bool(reward_gate["pass"].all())},
    {"family": "architecture", "all_gates_pass": bool(arch_gate["pass"].all())},
    {"family": "loss", "all_gates_pass": bool(loss_gate["pass"].all())},
])
display(gate_status)

if REQUIRE_COMPLETED_RESULTS:
    assert protocol_gate["pass"].all(), "Protocol gate failed"
    assert state_gate["pass"].all(), "State gate failed"
    assert reward_gate["pass"].all(), "Reward gate failed"
    assert arch_gate["pass"].all(), "Architecture gate failed"
    assert loss_gate["pass"].all(), "Loss gate failed"
else:
    display(Markdown("Hard assertions are disabled. Set `REQUIRE_COMPLETED_RESULTS = True` after you run all studies if you want the notebook to fail on any missing or unhealthy result family."))
